<a href="https://colab.research.google.com/github/leandroclv/polars_udemy/blob/main/lazyframe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl

In [2]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df = pl.read_csv(url)
df.head()

total_bill,tip,sex,smoker,day,time,size
f64,f64,str,str,str,str,i64
16.99,1.01,"""Female""","""No""","""Sun""","""Dinner""",2
10.34,1.66,"""Male""","""No""","""Sun""","""Dinner""",3
21.01,3.5,"""Male""","""No""","""Sun""","""Dinner""",3
23.68,3.31,"""Male""","""No""","""Sun""","""Dinner""",2
24.59,3.61,"""Female""","""No""","""Sun""","""Dinner""",4


In [6]:
df2 = df.with_columns([
    ((pl.col('total_bill')/(pl.col('tip') + pl.col('total_bill')))*100).round(2).alias('percent_gorjeta')
])

df2.select(['total_bill', 'tip', 'percent_gorjeta']).head()

total_bill,tip,percent_gorjeta
f64,f64,f64
16.99,1.01,94.39
10.34,1.66,86.17
21.01,3.5,85.72
23.68,3.31,87.74
24.59,3.61,87.2


In [10]:
df3 = df.with_columns([
    pl.lit('Brasil').alias('pais'),
    (pl.col('tip')*2).alias('tip_x2')
    ])
df3.select(['pais', 'tip', 'tip_x2']).head()

pais,tip,tip_x2
str,f64,f64
"""Brasil""",1.01,2.02
"""Brasil""",1.66,3.32
"""Brasil""",3.5,7.0
"""Brasil""",3.31,6.62
"""Brasil""",3.61,7.22


In [21]:
#Convertendo dataframe para lazyframe
lf = df.lazy()


#Construindo um pipeline
pipeline = (
    lf
    .filter(pl.col('total_bill') > 20)
    .with_columns((pl.col('tip')/pl.col('total_bill') * 100).round(2).alias('percent_gorjeta'))
    .group_by('day')
    .agg([
        pl.mean('percent_gorjeta').round(2).alias('media_percent_gorj'),
        pl.sum('total_bill').alias('total_tip')
    ])
    .sort('media_percent_gorj', descending=True)
)

#Executando com o collect()
result = pipeline.collect()
print(result)


shape: (4, 3)
┌──────┬────────────────────┬───────────┐
│ day  ┆ media_percent_gorj ┆ total_tip │
│ ---  ┆ ---                ┆ ---       │
│ str  ┆ f64                ┆ f64       │
╞══════╪════════════════════╪═══════════╡
│ Thur ┆ 14.74              ┆ 454.93    │
│ Sun  ┆ 14.56              ┆ 1057.54   │
│ Sat  ┆ 13.59              ┆ 1082.11   │
│ Fri  ┆ 13.49              ┆ 162.67    │
└──────┴────────────────────┴───────────┘
